# Step 1: Import Required Packages

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

C:\Users\Rohit singh\AppData\Local\Temp\ipykernel_14980\1987215824.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


# Step 2: Load Gemini API Key

In [10]:
import os

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    print("Gemini API key not found ❌")
else:
    llm = ChatGoogleGenerativeAI(
        model="gemini-3.6-flash",
        temperature=0.7
    )

    print("Gemini LLM initialized successfully")

Gemini LLM initialized successfully


# Step 3: Build Basic Chatbot

In [13]:
while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("AI: Goodbye! 👋")
        break

    response = llm.invoke(user_input)

    if isinstance(response.content, list):
        ai_text = "\n".join(
            item["text"]
            for item in response.content
            if isinstance(item, dict) and item.get("type") == "text"
        )
    else:
        ai_text = response.content

You:  hi


C:\Users\Rohit singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


You:  hello


C:\Users\Rohit singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


You:  exit


# Step 4: Conversation Memory

In [15]:
chat_history = []

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        # print("AI: Goodbye! 👋")
        break

    chat_history.append(
        HumanMessage(content=user_input)
    )

    response = llm.invoke(chat_history)

    if isinstance(response.content, list):
        ai_text = "\n".join(
            item["text"]
            for item in response.content
            if isinstance(item, dict) and item.get("type") == "text"
        )
    else:
        ai_text = response.content

    print("AI:", ai_text)

    chat_history.append(
        AIMessage(content=ai_text)
    )

You:  exit


AI: Goodbye! 👋


# Step 5: Add System Prompt

In [16]:
system_prompt = """
You are a helpful and friendly AI assistant.
Answer questions clearly and accurately.
If the user asks something you do not know, say that you are not sure.
Keep your answers easy to understand.
"""

response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content="What is Artificial Intelligence?")
])

if isinstance(response.content, list):
    ai_text = "\n".join(
        item["text"]
        for item in response.content
        if isinstance(item, dict) and item.get("type") == "text"
    )
else:
    ai_text = response.content

C:\Users\Rohit singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


# Step 6: System Prompt

In [17]:
system_prompt = """
You are a helpful and friendly AI assistant.

Your job is to:
- Answer questions clearly.
- Explain difficult topics in simple language.
- Give accurate and useful answers.
- If you don't know something, honestly say that you don't know.
- Be polite and professional.
"""

response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content="What is Artificial Intelligence?")
])

if isinstance(response.content, list):
    ai_text = "\n".join(
        item["text"]
        for item in response.content
        if isinstance(item, dict) and item.get("type") == "text"
    )
else:
    ai_text = response.content


C:\Users\Rohit singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


# Step 7: Combine System Prompt with Conversation Memory

In [18]:
system_prompt = """
You are a helpful and friendly AI assistant.

Your job is to:
- Answer questions clearly.
- Explain difficult topics in simple language.
- Give accurate and useful answers.
- Use the previous conversation when answering questions.
- If you don't know something, honestly say that you don't know.
- Be polite and professional.
"""

chat_history = [
    SystemMessage(content=system_prompt)
]

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("AI: Goodbye! 👋")
        break

    chat_history.append(
        HumanMessage(content=user_input)
    )

    response = llm.invoke(chat_history)

    if isinstance(response.content, list):
        ai_text = "\n".join(
            item["text"]
            for item in response.content
            if isinstance(item, dict) and item.get("type") == "text"
        )
    else:
        ai_text = response.content

    # print("AI:", ai_text)

    chat_history.append(
        AIMessage(content=ai_text)
    )

You:  exit


AI: Goodbye! 👋


AI: Goodbye! 👋


# Step 8: Create Chatbot Function

In [21]:
system_prompt = """
You are a helpful and friendly AI assistant.

Your job is to:
- Answer questions clearly.
- Explain difficult topics in simple language.
- Give accurate and useful answers.
- Use the previous conversation when answering questions.
- If you don't know something, honestly say that you don't know.
- Be polite and professional.
"""


def get_ai_response(chat_history):
    response = llm.invoke(chat_history)

    if isinstance(response.content, list):
        ai_text = "\n".join(
            item["text"]
            for item in response.content
            if isinstance(item, dict) and item.get("type") == "text"
        )
    else:
        ai_text = response.content

    return ai_text


def start_chat():
    chat_history = [
        SystemMessage(content=system_prompt)
    ]

    while True:
        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("AI: Goodbye! 👋")
            break

        chat_history.append(
            HumanMessage(content=user_input)
        )

        ai_text = get_ai_response(chat_history)

        print("AI:", ai_text)

        chat_history.append(
            AIMessage(content=ai_text)


        )


start_chat()

You:  hello


C:\Users\Rohit singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AI: Hello! How can I help you today?


You:  what is llm


C:\Users\Rohit singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


...

You:  exit


AI: Goodbye! 👋


You:  exits


C:\Users\Rohit singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AI: Goodbye, Shish! It was nice chatting with you. 

Have a wonderful day, and feel free to reach out if you need anything else in the future!
